In [1]:
qualtrics_fn = "qualtrics_test.xlsx"
ollama_fn = "../ollama-classifier/results/ollama_labels_fd56c7a07593ee29.xlsx"

In [2]:
import pandas as pd

df = pd.read_excel(qualtrics_fn, header=[0, 1])
df.columns = df.columns.get_level_values(0) # drop multi-index (contains full question)

substrings = ["1_", "2_", "3_", "4_", "5_"]
stacked_parts = []
columns = df.columns

for substring in substrings:
    mask = (
        columns.str.startswith(substring)  # current prefix
        &
        ~columns.str.contains("First Click|Last Click|Click Count", na=False)  # exclude these
    )
    filtered_df = df.loc[:,mask].copy()

    new_columns = filtered_df.columns.get_level_values(0).str.replace(
        f"^{substring}", "", regex=True
    )

    filtered_df.columns = new_columns

    filtered_df["Q1"] = df["Q1"]
    stacked_parts.append(filtered_df)

stacked_df = pd.concat(stacked_parts, axis=0, ignore_index=True)

# sum total time taken
columns = stacked_df.columns.get_level_values(0)
page_submit_mask = columns.str.contains("Page Submit", na=False)
page_submit_cols = stacked_df.loc[:, page_submit_mask]
final_df = stacked_df.loc[:, ~page_submit_mask]
final_df["time"] = page_submit_cols.sum(axis=1)



/opt/anaconda3/envs/validation/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [3]:
# alpha-numeric order for viewing (make sure we have all questions)
from natsort import natsorted

sorted_col_names = natsorted(final_df.columns)
df_sorted = final_df[sorted_col_names]
df_sorted.columns

Index(['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q8_5_TEXT', 'Q9',
       'Q10', 'Q11', 'Q11_13_TEXT', 'Q12', 'Q12_5_TEXT', 'Q13', 'Q13_5_TEXT',
       'Q14', 'Q14_5_TEXT', 'Q15', 'Q20', 'time'],
      dtype='str')

In [4]:
# Flatten multi-choice answers into single-feature columns

def bool_cols_from_multi_choice(df: pd.DataFrame, patterns: list[str], col: str) -> pd.DataFrame:
    for p in patterns:
        _col_name = p.lower().replace(" ", "_")
        df[_col_name] = df[col].str.contains(p, case=False, na=False, regex=False)
    return df

patterns = [
    [
    "GRDA",
    "GPD",
    "GPD with triphasic morphology",
    "Hypsarrythmia"
    ], 
    [
    "Generalized spike and wave (<3 Hz)",
    "Generalized spike and wave (3-4 Hz)",
    "Generalized spike and wave (>4 Hz)",
    "Generalized Paroxsymal Fast Activity"
    ],
    [
    "Sharps",
    "Spikes",
    "LPDs",
    "LRDA/TIRDA",
    "Focal paroxysmal fast"
    ],
    [
    "Ictal-Interictal-Continuum (IIC)",
    "Brief (potentially ictal) Rhythmic Discharges (BiRDs)",
    "Epileptic Seizure",
    "Status Epilepticus",
    "Functional (Nonepileptic) Seizure",
    "Non-Functional Non-Epileptic Event"
    ],
    [
    "Focal",
    "Generalized",
    "Atonic/tonic",
    "Spasm"
    ],
    [
    "Hypermotor - big movements",
    "Positive signs observable to others",
    "Subjective with lapse or alteration of consciousness",
    "Subjective symptoms only"
    ],
    [
    "Nonconvulsive",
    "Convulsive",
    "Status Triphasicus",
    "Focal Status"
    ],
    [
    "Migraine or headache",
    "Pre-syncope, Syncope, or other cardiovascular",
    "Behavioral non-functional",
    "Accidental or other event of non-interest",
    ],
    [
    "Focal slowing",
    "Focal sharply contoured activity",
    "Asymmetric background activity or sleep morphology",
    ]
]

cols = {
    "Q6":str, "Q8":str,
    "Q9":str, "Q10":str,
    "Q11":str, "Q12":str, 
    "Q13":str, "Q14":str,
    "Q20":str
    }


df_sorted = df_sorted.astype(cols)

for pattern, col in zip(patterns, cols):
    extracted_df = bool_cols_from_multi_choice(df_sorted, pattern, col)

extracted_df = extracted_df.drop(columns=cols.keys())
extracted_df.columns

Index(['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q7', 'Q8_5_TEXT', 'Q11_13_TEXT',
       'Q12_5_TEXT', 'Q13_5_TEXT', 'Q14_5_TEXT', 'Q15', 'time', 'grda', 'gpd',
       'gpd_with_triphasic_morphology', 'hypsarrythmia',
       'generalized_spike_and_wave_(<3_hz)',
       'generalized_spike_and_wave_(3-4_hz)',
       'generalized_spike_and_wave_(>4_hz)',
       'generalized_paroxsymal_fast_activity', 'sharps', 'spikes', 'lpds',
       'lrda/tirda', 'focal_paroxysmal_fast',
       'ictal-interictal-continuum_(iic)',
       'brief_(potentially_ictal)_rhythmic_discharges_(birds)',
       'epileptic_seizure', 'status_epilepticus',
       'functional_(nonepileptic)_seizure',
       'non-functional_non-epileptic_event', 'focal', 'generalized',
       'atonic/tonic', 'spasm', 'hypermotor_-_big_movements',
       'positive_signs_observable_to_others',
       'subjective_with_lapse_or_alteration_of_consciousness',
       'subjective_symptoms_only', 'nonconvulsive', 'convulsive',
       'status_triphasicus', 

In [ ]:
# Rename single-feature columns
single_feat_cols = {
    "Q1":"annotator",
    "Q2":"patient_setting", 
    "Q3":"duration",
    "Q4":"abnormal",
    "Q5":"encephalopathy_modifier",
    "Q7": "focal_slowing_non_epil_location",
    "Q8":"gen_epil",
    "Q8_5_TEXT":"gen_epil_txt",
    "Q11_13_TEXT":"epil_sz_type_txt",
    "Q12_5_TEXT":"func_sz_type_txt",
    "Q13_5_TEXT":"se_type_txt",
    "Q14_5_TEXT":"non_epil_non_func_type_txt",
    "Q15":"note_id"
    }
extracted_df.rename(columns=single_feat_cols, inplace=True)

# Ensure open-text 'other' fields are string type
new_dtypes = {}
for value in single_feat_cols.values():
    if "txt" in value:
        new_dtypes[value] = str

extracted_df["abnormal"] = extracted_df["abnormal"] == "Abnormal"
extracted_df = extracted_df.astype(new_dtypes)
extracted_df.dtypes



dtype('float64')

In [6]:
# some more quality control
# check annotator names; see if any mislabeled
print(extracted_df["annotator"].value_counts()) # see if any incorrect names
extracted_df.loc[extracted_df['annotator'] == "3426223636", "annotator"] = 'Hoameng' #3426223636 should be HU

# Handle missing data in note_id
extracted_df = extracted_df.dropna(subset='note_id')

# Process open-text questions (remove whitespace)
extracted_df["note_id"] = extracted_df["note_id"].astype(int)
cols_to_strip = [
    "annotator", "gen_epil_txt", 
    "epil_sz_type_txt", "func_sz_type_txt", 
    "se_type_txt", "non_epil_non_func_type_txt"
    ]
extracted_df[cols_to_strip] = extracted_df[cols_to_strip].apply(lambda x: x.str.strip())

# filter out 'test' annotations
extracted_df = extracted_df[~extracted_df["annotator"].str.lower().str.contains('test')]

# find duplicate note_id's; if there are duplicates, keep the ones with "HU" as the annotator name
print(extracted_df.value_counts('note_id') [extracted_df.value_counts('note_id') > 1]) # see duplicate id's

extracted_df = (
    extracted_df.sort_values(by="annotator")  # sort by annotator
      .sort_values(by="annotator", key=lambda x: x != "Hoameng")  # Hoameng = False, and in sorting are first
      .drop_duplicates(subset="note_id", keep="first") # keep first (HU)
)

print(extracted_df.value_counts('note_id') [extracted_df.value_counts('note_id') > 1])

df_cleaned = extracted_df

df_cleaned.shape[0]


annotator
Zirui Song    35
Hoameng       25
Solana        25
test           5
3426223636     5
Name: count, dtype: int64
note_id
1467038687    2
Name: count, dtype: int64
Series([], Name: count, dtype: int64)


80

In [7]:
q_features = [
    "note_id", 
    "epileptic_seizure",                   # SZ 
    "status_epilepticus",                  # SE
    "nonconvulsive",                       # NCSE
    "encephalopathy_modifier",             # Diffuse Abnormalities (as long as not "No identifiable cerebrally generated rhythms")
    "focal_slowing"
]

q_epil_discharge_features = [
    "generalized_spike_and_wave_(<3_hz)", # epileptiform discharges (general or focal)
    "generalized_spike_and_wave_(3-4_hz)",
    "generalized_spike_and_wave_(>4_hz)",
    "generalized_paroxsymal_fast_activity",
    "sharps",
    "spikes",
    "lpds",
    "lrda/tirda",
    "focal_paroxysmal_fast",
    "gen_epil_txt"                                    
]

df_cleaned["epileptiform_discharges"] = df_cleaned[q_epil_discharge_features].any(axis='columns')
df_cleaned["diffuse_nonspecific_abnormalities"] = df_cleaned["encephalopathy_modifier"] != "No identifiable cerebrally generated rhythms"
df1 = df_cleaned[q_features + ["diffuse_nonspecific_abnormalities", "epileptiform_discharges"]]

In [8]:
# Read in classifier results
ollama_out = pd.read_excel(ollama_fn)
ollama_out["note_id"].dtype
merged_df = ollama_out.merge(df1, on="note_id",suffixes=["_ollama","_val"])

In [ ]:
# Overall T/F Accuracy
sz_acc = merged_df[merged_df["epileptic_seizure"] == merged_df["seizure"]].shape[0] / merged_df.shape[0]
se_acc = merged_df[merged_df["status_epilepticus_ollama"] == merged_df["status_epilepticus_val"]].shape[0] / merged_df.shape[0]
ncse_acc = merged_df[merged_df["ncse"] == merged_df["nonconvulsive"]].shape[0] / merged_df.shape[0]
ed_acc = merged_df[merged_df["epileptiform_discharges_ollama"] == merged_df["epileptiform_discharges_val"]].shape[0] / merged_df.shape[0]
dna_acc = merged_df[merged_df["diffuse_nonspecific_abnormalities_ollama"] == merged_df["diffuse_nonspecific_abnormalities_val"]].shape[0] / merged_df.shape[0]

In [10]:
print(sz_acc, se_acc, ncse_acc, ed_acc, dna_acc)

0.9142857142857143 0.9857142857142858 0.9857142857142858 0.8571428571428571 0.8857142857142857


In [14]:
merged_df[merged_df["ncse"] == True]

,run_id,note_id,elapsed_s,seizure,status_epilepticus_ollama,ncse,epileptiform_discharges_ollama,diffuse_nonspecific_abnormalities_ollama,focal_slowing_ollama,epileptic_seizure,status_epilepticus_val,nonconvulsive,encephalopathy_modifier,focal_slowing_val,diffuse_nonspecific_abnormalities_val,epileptiform_discharges_val
31,fd56c7a07593ee29,2720795200,9.343621,False,False,True,False,True,0.0,False,False,False,NaN,False,True,False
